# Project 1: Company Sales Brochure Generator
- Using OpenAI API
- One-shot prompting
- stream back results and show with formatting

In [5]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

#initialization and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
'''if api_key:
    print("API key looks good so far")'''
MODEL = 'gpt-5-nano'
openai = OpenAI()

links = fetch_website_links("https://edwarddonner.com") #gets all links from page
#can give relative or absolute links

### Step 1 -- find relevant links using gpt

In [ ]:
#need to figure out which links are relevant to what we want -- use openai call
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""
#above: one-shot prompting -- giving it an example of what you want so it can learn from it

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
    
Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

print(get_links_user_prompt("https://edwarddonner.com"))

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}...")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content":link_system_prompt},
            {"role":"user", "content":get_links_user_prompt(url)}
        ],
        response_format = {"type":"json_object"} #another parameter - tell the API we want response to be in a particular format
    )
    result = response.choices[0].message.content
    links = json.loads(result) #turn strings into dictionaries
    print(f"Found {len(links['links'])} relevant links")
    return links 

select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by colling gpt-5-nano...
Found 5 relevant links


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by colling gpt-5-nano...
Found 12 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community / Discord',
   'url': 'https://huggingface.co/join/discord'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Discussions', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Status', 'url': 'https://status.huggingface.co/'},
  {'type': 'Endpoints product', 'url': 'https://endpoints.huggingface.co'}]}

### Step 2 -- make the brochure

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url) #get info on page
    relevant_links = select_relevant_links(url) #part 1
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']: #for all the links we got,
        result += f"\n\n### Link: {link['type']}\n" #put the link
        result += fetch_website_contents(link["url"])  #and the contents of the link website
    return result

print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt="""
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

get_brochure_user_prompt("HuggingFace","https://huggingface.co")